# disent_from_attn_out — disentangle q / kv transfer from o-transfer

**Problem.** Our standard q-transfer and kv-transfer setups both make DST pursue `SRC.targ` — but so
does an **o-transfer** (transplanting the goal heads' ATTN-OUT / OV contribution, i.e. the *output* of
the DIFF read). So the standard setups can't tell the three pathways apart: q, kv, and o all point at
`SRC.targ`.

**Fix.** Design setups where the three pathways predict **different** tokens.

| section | q / kv transfer → | o-transfer → | clean DST → |
|---|---|---|---|
| **q-transfer** | **T4** (progress pointer resolved against DST's *own* list) | **T3** (`SRC.targ`, resolved goal) | T2 |
| **kv-transfer** | **T2** (SRC task *menu* resolved at DST's progress) | **T3** (`SRC.targ`) | T4 |

- **q-transfer.** SRC(T1,T2,T3), prefill{T1,T2}, targ **T3**;  DST(T1,T2,T4), prefill{T1}, clean-targ **T2**.
  If the query carries *progress* ("T1,T2 done"), transplanting it into DST (which has only T1 done)
  should make DST skip to **T4** (its own 3rd slot). The OV output carries the resolved goal → **T3**.
- **kv-transfer.** SRC(T1,T2,T3), prefill{T1,T2}, targ **T3**;  DST(T1,T4,T5), prefill{T1}, clean-targ **T4**.
  If the keys/values carry the *task list* {T1,T2,T3}, transplanting it into DST (T1 done) should make
  DST do **T2** (the SRC menu's next-after-T1). The OV output → **T3**.

Since **T4 ∉ SRC** and **T2 ≠ T3**, each pathway has a unique signature. Qualitative check only.

Utils exercised: pad-aligned k/v span transplant (kv), attn-gaze debugging (q), GoalScope readout (both),
and a small o_proj-input replacement helper for o-transfer.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0, 2")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append("../..")
import torch, numpy as np, baukit
from contextlib import contextmanager, nullcontext
import src.diff.functional as P
from src.diff.functional import *          # public helpers into scope
import src.diff.data as D
from src.attention import get_attention_matrices
from rich.panel import Panel
from rich.text import Text
from rich.markup import escape

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct")  # loads model + head masks (once)
mt = P.mt
GOAL_HEADS, UNION_HEADS, GROUP = P.GOAL_HEADS, P.UNION_HEADS, P.GROUP
PATCH_HEADS = GOAL_HEADS                     # 122 goal heads. To use top-250 instead, see next cell.
ALPHA = 2.0                                  # OV-write boost of PATCH_HEADS at the pivot on the PATCHED gen/readout (1.0 = off)
O_ALPHA = 1.0                                # o-transfer (GoalScope 'replace') source-side weight (1.0 = full replace)
print(f"PATCH_HEADS = {len(PATCH_HEADS)} heads")

2026-09-07 22:31:37 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228
PATCH_HEADS = 122 heads


In [2]:
# # OPTIONAL — use the top-250 goal-head set (patch + readout), which gave near-perfect kv transfer.
# import json
# _p = "../../head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json"
# PATCH_HEADS = [tuple(h) for h in json.load(open(_p))]
# # mt is unchanged; goalscope readout uses ctx.readout_heads unless you pass heads=PATCH_HEADS (we do).
# print("PATCH_HEADS =", len(PATCH_HEADS))

In [3]:
# ── helpers (prompt display, instance builder, o-transfer via o_proj-input replacement) ──
def pivot_of(prompt, neg_shift=-1):
    tok = prepare_input(prompts=prompt, tokenizer=mt) if isinstance(prompt, str) else prompt
    return tok["input_ids"].shape[1] + neg_shift

def show(prompt, title, emph_span=None, piv=-1):
    kw = dict(patch_pos=[piv])
    if emph_span is not None:
        kw["emphasize_tokens"] = range(*emph_span)
    rich_print(Panel(Text.from_markup(exact_prompt(prompt, **kw)),
                     title=f"[bold]{escape(title)}[/bold]", title_align="left",
                     border_style="grey42", padding=(1, 1)))

BANK = D.QTASKS["list[str]"]; BY = {p.id: p for p in BANK.predicates}
# five distinct, pairwise-compatible list[str] predicates
T1, T2, T3, T4, T5 = "total_length", "count_long", "count_starts_vowel", "has_palindrome", "distinct_count"
NAMING = "letters"

def mk(pred_ids, progress, target):
    """A QTransferInstance over an arbitrary predicate list (listed presentation, letters naming).
    We only ever read src_parts() of a SRC-instance or dst_parts() of a DST-instance."""
    task = D.dataclasses.replace(BANK, id="xtask", predicates=tuple(BY[i] for i in pred_ids))
    return D.QTransferInstance(task=task, var_name=BANK.var_names[0], presentation="listed",
        src_lang="python", dst_lang="python", src_progress=progress, dst_progress=progress,
        expected_next=target, naming=NAMING, meta={"datatype": "list[str]"})

def list_body(inst):
    """The numbered task-list body exactly as _question renders it (the kv transplant span)."""
    return "\n".join(f"({i+1}) {p.description}" for i, p in enumerate(inst.task.predicates))

def ov_boost(positions=(-1,)):
    """OV-write boost of PATCH_HEADS by ALPHA at `positions` (nullcontext when ALPHA==1.0)."""
    return boost_head_ov(mt, PATCH_HEADS, ALPHA, positions=positions) if ALPHA != 1.0 else nullcontext()

# ── o-transfer = GoalScope strategy="replace", PERSISTENT across ALL decode steps ──
# The EXACT GoalScope replace intervention: replace PATCH_HEADS' o_proj INPUT (per-head attn output)
# at the destination's last token with SRC's (extracted at the SRC pivot), re-applied EVERY generation
# step (n_steer_tokens="all"). We point the GoalScope dest at the DST *code* prompt, so it generates
# DST's continuation. O_ALPHA = replace source-side weight (1.0 = full replace).
from src.goals.goalscope import apply_goalscope

def o_transfer_gen(src_prompt, dst_prompt, src_pivot, heads, alpha=None, max_new=96):
    a = O_ALPHA if alpha is None else alpha
    res = apply_goalscope(prompt=src_prompt, mt=mt, selected_heads=list(heads), strategy="replace",
                          alpha=a, goalscope_prompt=dst_prompt, token_positions=[src_pivot],
                          n_steer_tokens="all", max_new_tokens=max_new, show=False)
    return res[0]["patched_gen"]

def show_spans(tok, title, task_span=None, kv_span=None, piv=-1):
    """Token-aware prompt renderer (for pad-aligned TokenizerOutputs). task_span -> bright_yellow;
    kv_span -> black on dark_orange (the EXACT transplant window); pivot -> salmon. Title is escaped."""
    ids = (prepare_input(prompts=tok, tokenizer=mt) if isinstance(tok, str) else tok)["input_ids"][0].tolist()
    n = len(ids); pivpos = n + piv if piv < 0 else piv
    task = set(range(*task_span)) if task_span else set()
    kv = set(range(*kv_span)) if kv_span else set()
    PO, PC, KO, KC, TO, TC = "\x00<", "\x00>", "\x01<", "\x01>", "\x02<", "\x02>"
    parts = []
    for i in range(n):
        s = mt.tokenizer.decode([ids[i]]).replace("\n", "\\n\n")
        if i == pivpos:   s = PO + s + PC
        elif i in kv:     s = KO + s + KC
        elif i in task:   s = TO + s + TC
        parts.append(s)
    txt = escape("".join(parts))
    txt = (txt.replace(PO, "[bold black on salmon1]").replace(PC, "[/bold black on salmon1]")
              .replace(KO, "[bold black on dark_orange]").replace(KC, "[/bold black on dark_orange]")
              .replace(TO, "[bright_yellow]").replace(TC, "[/bright_yellow]"))
    rich_print(Panel(Text.from_markup(txt), title=f"[bold]{escape(title)}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))

# NOTEBOOK-LOCAL instruction tweak (does NOT touch the data pipeline): every prompt built below reads
# "computes in sequential order:" instead of "computes:". Shadows get_prompt for this notebook only.
_base_get_prompt = get_prompt
def get_prompt(question, prefill, ctx=None):
    return _base_get_prompt(question.replace("computes:", "computes in sequential order:"), prefill, ctx=ctx)

print("helpers ready ·", {k: BY[k].description for k in (T1,T2,T3,T4,T5)})

helpers ready · {'total_length': 'the total number of characters across all words', 'count_long': 'the number of words longer than five characters', 'count_starts_vowel': 'the number of words that start with a vowel', 'has_palindrome': 'whether any word is a palindrome', 'distinct_count': 'the number of distinct words'}


## 1. q-transfer  (query ⇒ progress)  — disentangle from o-transfer

SRC(T1,T2,T3) prefill{T1,T2} → about to write **T3**.  DST(T1,T2,T4) prefill{T1} → about to write **T2**.

- **query transplant** should trigger **T4** (SRC's progress "T1,T2 done" resolved against DST's own list).
- **o-transfer** (goal-head attn-out) should trigger **T3** (the resolved SRC goal).
- clean DST writes **T2**.

In [4]:
# build SRC / DST prompts for the q-transfer disentanglement
q_src = mk([T1, T2, T3, T4], [T1, T2], T3)     # SRC: T1,T2 done → target T3
q_dst = mk([T1, T2, T4, T5, T3], [T1],     T2)     # DST: T1 done   → frontier T2 (its 3rd slot is T4)
_q, _pf = q_src.src_parts(); qsrc_prompt = get_prompt(_q, D._finalize_prefill(_pf))
_q, _pf = q_dst.dst_parts(); qdst_prompt = get_prompt(_q, D._finalize_prefill(_pf))
qpiv_src, qpiv_dst = pivot_of(qsrc_prompt), pivot_of(qdst_prompt)
print(f"T2={BY[T2].description!r}  T3={BY[T3].description!r}  T4={BY[T4].description!r}")
show(qsrc_prompt, f"SRC · [{T1},{T2},{T3}] · {{{T1},{T2}}} done · targ {T3} · pivot@{qpiv_src}")
show(qdst_prompt, f"DST · [{T1},{T2},{T4}] · {{{T1}}} done · clean-targ {T2} · pivot@{qpiv_dst}")

qsrc_tok = prepare_input(prompts=qsrc_prompt, tokenizer=mt)
qdst_tok = prepare_input(prompts=qdst_prompt, tokenizer=mt)

T2='the number of words longer than five characters'  T3='the number of words that start with a vowel'  T4='whether any word is a palindrome'


╭─ SRC · [total_length,count_long,count_starts_vowel] · {total_length,count_long} done · targ count_starts_vowel ─╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes in    │
│ sequential order:\n                                                                                             │
│ (1) the total number of characters across all words\n                                                           │
│ (2) the number of words longer than five characters\n                                                           │
│ (3) the number of words that start with a vowel\n                                                               │
│ (4) whether any word is a palindrome\n                                                                          │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(len(w) for w in words)\n                                                                            │
│     b = sum(1 for w in words if len(w) > 5)\n                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · [total_length,count_long,has_palindrome] · {total_length} done · clean-targ count_long · pivot@149 ──────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes in    │
│ sequential order:\n                                                                                             │
│ (1) the total number of characters across all words\n                                                           │
│ (2) the number of words longer than five characters\n                                                           │
│ (3) whether any word is a palindrome\n                                                                          │
│ (4) the number of distinct words\n                                                                              │
│ (5) the number of words that start with a vowel\n                                                               │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(len(w) for w in words)\n                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [5]:
# clean baselines + GoalScope readouts
rich_print(f"[red1]clean SRC (expect {T3!r}):[/red1]\n{escape(greedy(qsrc_tok, max_new=96))}")
goalscope_readout("   ↳ goalscope · SRC", qsrc_prompt, qpiv_src, heads=PATCH_HEADS)
rich_print(f"[sea_green3]clean DST (expect {T2!r}):[/sea_green3]\n{escape(greedy(qdst_tok, max_new=96))}")
goalscope_readout("   ↳ goalscope · DST", qdst_prompt, qpiv_dst, heads=PATCH_HEADS)

clean SRC (expect 'count_starts_vowel'):
 c = sum(1 for w in words if w[0].lower() in 'aeiou')
    d = any(w == w[::-1] for w in words)
    return a, b, c, d
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


   ↳ goalscope · SRC  pos=158 '    ' => I'm checking if the character is a vowel.

clean DST (expect 'count_long'):
 b = sum(1 for w in words if len(w) > 5)
    c = any(w == w[::-1] for w in words)
    d = len(set(words))
    e = sum(1 for w in words if w[0].lower() in 'aeiou')
    return a, b, c, d, e
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · DST  pos=149 '    ' => I'm checking if the current element is longer than 5 characters.

[{'pos': 149,
  'token': '    ',
  'patched_gen': "I'm checking if the current element is longer than 5 characters.",
  'scoped': True}]

In [6]:
# (A) q-transfer: transplant SRC pivot QUERY → DST pivot   [expect T4]
qQ_LAYERS = sorted({l for l, _ in PATCH_HEADS})
q_cache = cache_src_proj(qsrc_prompt, [qpiv_src], qQ_LAYERS, "q")
_qh = heads_per_layer(PATCH_HEADS, "q")
qpf = [dict(which="q", cache=q_cache, heads=_qh, positions=[qpiv_dst])]
qdc = [dict(which="q", cache=q_cache, heads=_qh, positions=[-1])]
q_iface = make_patch_interface(mt, qpf)                 # PLAIN patch (used for gaze — boost scales the write, not attention)
with ov_boost():
    qgen = greedy(qdst_tok, prefill_specs=qpf, decode_specs=qdc, n_steer=32, max_new=96)
def q_patched_iface():                                   # query patch (+ ALPHA boost) for the readout
    if ALPHA == 1.0:
        return q_iface
    def _if(mt, inputs, patches=None, model_kwargs=None):
        with proj_patch_hooks(mt, qpf), ov_boost():
            return mt._model(**dict(inputs), **dict(model_kwargs or {}))
    return _if
rich_print(f"[salmon1]q-transfer · DST (expect T4={T4!r}):[/salmon1]\n{escape(qgen)}")
goalscope_readout("   ↳ goalscope · q-transfer", qdst_prompt, qpiv_dst, heads=PATCH_HEADS, iface=q_patched_iface())

# gaze debugging: value-weighted pivot→DST-TASK attention, clean vs q-patched (SRC target T3 not in DST)
q_spans = [token_span(qdst_prompt, p.description) for p in q_dst.task.predicates]
qTASK = list(range(min(s[0] for s in q_spans), max(s[1] for s in q_spans)))
qtoks = [mt.tokenizer.decode([int(qdst_tok["input_ids"][0][j])]) for j in qTASK]
q_clean_i = get_attention_matrices(qdst_tok, mt, value_weighted=True)
q_patch_i = get_attention_matrices(qdst_tok, mt, value_weighted=True, patch_interface=q_iface)
a_c = pivot_attn(q_clean_i, list(PATCH_HEADS), qTASK, qpiv_dst)
a_p = pivot_attn(q_patch_i, list(PATCH_HEADS), qTASK, qpiv_dst)
show_delta_tokens(qtoks, a_p - a_c, "gaze Δ · q-patched − clean (pivot→DST-TASK, PATCH_HEADS)")

q-transfer · DST (expect T4='has_palindrome'):
 b = sum(1 for w in words if w[0].lower() in 'aeiou')
    c = sum(1 for w in words if len(w) > 5)
    d = any(w == w[::-1] for w in words)
    e = len(set(words))
    return a, c, d, e, b
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · q-transfer  pos=149 '    ' => I'm checking if the current character is a vowel.

gaze Δ · q-patched − clean (pivot→DST-TASK, PATCH_HEADS)   (green +Δ / red −Δ, |Δ|max=1.620)

In [7]:
# ── CONTROL: transplant the QUERY of ALL heads on an EARLY layer band (vs the localized goal heads) ──
# Does the q-transfer → T4 effect need the goal heads, or does any early-layer query transplant produce it?
early_patch_heads = [(l, h) for l in range(30, 45) for h in range(mt.n_heads)]   # ALL heads, layers 20–29
early_layers = sorted({l for l, _ in early_patch_heads})
early_q_cache = cache_src_proj(qsrc_prompt, [qpiv_src], early_layers, "q")
eqh = heads_per_layer(early_patch_heads, "q")
eqpf = [dict(which="q", cache=early_q_cache, heads=eqh, positions=[qpiv_dst])]   # prefill
eqdc = [dict(which="q", cache=early_q_cache, heads=eqh, positions=[-1])]         # decode re-clamp
e_iface = make_patch_interface(mt, eqpf)
# plain query transplant, no OV boost — ALPHA boosts the GOAL heads; boosting this whole early band would derail gen.
egen = greedy(qdst_tok, prefill_specs=eqpf, decode_specs=eqdc, n_steer=32, max_new=96)
rich_print(f"[salmon1]early q-transfer · layers {early_layers[0]}\u2013{early_layers[-1]} (ALL {mt.n_heads} heads/layer) · DST (expect T4={T4!r}):[/salmon1]\n{escape(egen)}")
goalscope_readout("   \u21b3 goalscope · early q-transfer", qdst_prompt, qpiv_dst, heads=PATCH_HEADS, iface=e_iface)

# gaze: value-weighted pivot\u2192DST-TASK attention over the EARLY heads, clean vs patched
q_spans = [token_span(qdst_prompt, p.description) for p in q_dst.task.predicates]
qTASK = list(range(min(s[0] for s in q_spans), max(s[1] for s in q_spans)))
qtoks = [mt.tokenizer.decode([int(qdst_tok["input_ids"][0][j])]) for j in qTASK]
e_clean_i = get_attention_matrices(qdst_tok, mt, value_weighted=True)
e_patch_i = get_attention_matrices(qdst_tok, mt, value_weighted=True, patch_interface=e_iface)
a_c = pivot_attn(e_clean_i, list(early_patch_heads), qTASK, qpiv_dst)
a_p = pivot_attn(e_patch_i, list(early_patch_heads), qTASK, qpiv_dst)
show_delta_tokens(qtoks, a_p - a_c, f"gaze \u0394 · early-q-patched \u2212 clean (pivot\u2192DST-TASK, layers {early_layers[0]}\u2013{early_layers[-1]})")

early q-transfer · layers 30–44 (ALL 40 heads/layer) · DST (expect T4='has_palindrome'):
 b = sum(1 for w in words if len(w) > 5)
    c = any(w == w[::-1] for w in words)
    d = len(set(words))
    e = sum(w[0].lower() in 'aeiou' for w in words)
    return a, b, c, d, e
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · early q-transfer  pos=149 '    ' => Checking if any element in the list is longer than 5 
characters.

gaze Δ · early-q-patched − clean (pivot→DST-TASK, layers 30–44)   (green +Δ / red −Δ, |Δ|max=0.219)

In [8]:
# (B) o-transfer = GoalScope replace (persistent, every step): SRC goal-head attn-out → DST   [expect T3]
qgen_o = o_transfer_gen(qsrc_prompt, qdst_prompt, qpiv_src, PATCH_HEADS, max_new=96)
rich_print(f"[orange3]o-transfer · DST (expect T3={T3!r}):[/orange3]\n{escape(qgen_o)}")
# (a goalscope readout under o-transfer trivially returns the SRC goal — it IS the injected write;
#  see the clean-SRC readout above.)

print(f"\nSUMMARY (q-section) — clean→T2({T2}) · q-transfer→expect T4({T4}) · o-transfer→expect T3({T3})")

GoalScope[replace]:   0%|          | 0/1 [00:00<?, ?it/s]

o-transfer · DST (expect T3='count_starts_vowel'):
 b = sum(w[0].lower() in 'aeiou' for w in words)
    c = sum(w[0].lower() in 'aeiou' for w in words)
    d = sum(w[0].lower() in 'aeiou' for w in words)
    e = sum(w[0].lower() in 'aeiou' for w in words)
    f = sum(w[0].lower() in 'aeiou' for w


SUMMARY (q-section) — clean→T2(count_long) · q-transfer→expect T4(has_palindrome) · o-transfer→expect T3(count_starts_vowel)


## 2. kv-transfer  (keys/values ⇒ goal)  — disentangle from o-transfer

SRC(T1,T2,T3) prefill{T1,T2} → about to write **T3**.  DST(T1,T4,T5) prefill{T1} → about to write **T4**.

- **k/v transplant** of the SRC task-list span should trigger **T2** (SRC menu {T1,T2,T3}, DST has T1 done → next = T2).
- **o-transfer** should trigger **T3** (resolved SRC goal).
- clean DST writes **T4**.

Uses **pad**-aligned k/v transplant over the numbered task-list span (`align_kv_spans`), GQA-corrected
headwise routing (`headwise_kv_patch`).

In [9]:
# build SRC / DST prompts; locate + pad-align the task-list spans
k_src = mk([T1, T2, T3], [T1, T2], T3)     # SRC: T1,T2 done → target T3
k_dst = mk([T1, T4, T5], [T1],     T4)     # DST: T1 done   → frontier T4
_q, _pf = k_src.src_parts(); ksrc_prompt = get_prompt(_q, D._finalize_prefill(_pf))
_q, _pf = k_dst.dst_parts(); kdst_prompt = get_prompt(_q, D._finalize_prefill(_pf))
src_text, dst_text = list_body(k_src), list_body(k_dst)     # transplant span = task-list body
src_span0 = token_span(ksrc_prompt, src_text)
dst_span0 = token_span(kdst_prompt, dst_text)
ksrc_tok, kdst_tok, src_pos, dst_pos, src_span, dst_span = align_kv_spans(
    mt, ksrc_prompt, kdst_prompt, src_span0, dst_span0, mode="pad",
    src_text=src_text, dst_text=dst_text)
kpiv_src = ksrc_tok["input_ids"].shape[1] - 1
kpiv_dst = kdst_tok["input_ids"].shape[1] - 1
print(f"T2={BY[T2].description!r}  T3={BY[T3].description!r}  T4={BY[T4].description!r}")
show_spans(ksrc_tok, f"SRC · [{T1},{T2},{T3}] · cached-from KV window {tuple(src_span)} · pivot@{kpiv_src}",
           task_span=src_span, kv_span=(src_pos[0], src_pos[-1] + 1), piv=-1)
show_spans(kdst_tok, f"DST · [{T1},{T4},{T5}] · patched-to KV window {tuple(dst_span)} · pivot@{kpiv_dst}",
           task_span=dst_span, kv_span=(dst_pos[0], dst_pos[-1] + 1), piv=-1)

T2='the number of words longer than five characters'  T3='the number of words that start with a vowel'  T4='whether any word is a palindrome'


╭─ SRC · [total_length,count_long,count_starts_vowel] · cached-from KV window (66, 102) · pivot@148 ──────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes in    │
│ sequential order:\n                                                                                             │
│ (1) the total number of characters across all words\n                                                           │
│ (2) the number of words longer than five characters\n                                                           │
│ (3) the number of words that start with a vowel\n                                                               │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(len(w) for w in words)\n                                                                            │
│     b = sum(1 for w in words if len(w) > 5)\n                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · [total_length,has_palindrome,distinct_count] · patched-to KV window (66, 102) · pivot@130 ───────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes in    │
│ sequential order:\n                                                                                             │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ (1) the total number of characters across all words\n                                                           │
│ (2) whether any word is a palindrome\n                                                                          │
│ (3) the number of distinct words\n                                                                              │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(len(w) for w in words)\n                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [10]:
# clean baselines + GoalScope readouts
rich_print(f"[red1]clean SRC (expect {T3!r}):[/red1]\n{escape(greedy(ksrc_tok, max_new=96))}")
goalscope_readout("   ↳ goalscope · SRC", ksrc_tok, kpiv_src, heads=PATCH_HEADS)
rich_print(f"[sea_green3]clean DST (expect {T4!r}):[/sea_green3]\n{escape(greedy(kdst_tok, max_new=96))}")
goalscope_readout("   ↳ goalscope · DST", kdst_tok, kpiv_dst, heads=PATCH_HEADS)

clean SRC (expect 'count_starts_vowel'):
 c = sum(1 for w in words if w[0].lower() in 'aeiou')
    return a, b, c
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · SRC  pos=148 '    ' => I'm checking if the character is a vowel.

clean DST (expect 'has_palindrome'):
 b = any(w == w[::-1] for w in words)
    c = len(set(words))
    return a, b, c
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · DST  pos=130 '    ' => I'm checking if any element in the list is equal to its reverse (indicating
a palindrome).

[{'pos': 130,
  'token': '    ',
  'patched_gen': "I'm checking if any element in the list is equal to its reverse (indicating a palindrome).",
  'scoped': True}]

In [11]:
# (A) kv-transfer: transplant SRC task-list KEYS+VALUES → DST task span   [expect T2]
kvheads = heads_per_layer(PATCH_HEADS, "k", layers=set(P._layers_of(PATCH_HEADS)))
k_cache = cache_src_proj(ksrc_tok, src_pos, sorted(kvheads), "k")
v_cache = cache_src_proj(ksrc_tok, src_pos, sorted(kvheads), "v")
def kv_patched_iface():                                  # k/v transplant (+ ALPHA boost) for the readout
    def _if(mt, inputs, patches=None, model_kwargs=None):
        with headwise_kv_patch(mt, PATCH_HEADS, k_cache, v_cache, dst_pos), ov_boost():
            return mt._model(**dict(inputs), **dict(model_kwargs or {}))
    return _if
with headwise_kv_patch(mt, PATCH_HEADS, k_cache, v_cache, dst_pos), ov_boost():
    kvgen = greedy(kdst_tok, max_new=96)
rich_print(f"[salmon1]kv-transfer · DST (expect T2={T2!r}):[/salmon1]\n{escape(kvgen)}")
goalscope_readout("   ↳ goalscope · kv-transfer", kdst_tok, kpiv_dst, heads=PATCH_HEADS, iface=kv_patched_iface())

kv-transfer · DST (expect T2='count_long'):
 b = sum(1 for w in words if w[0].lower() in 'aeiou')
    c = sum(1 for w in words if len(w) > 5)
    return a, b, c

# Example usage:
# words = ["apple", "banana", "cherry", "date"]
# result = func(words)
# print(result)  # Output: (19, 2, 2)
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · kv-transfer  pos=130 '    ' => I'm checking if the character is a vowel.

[{'pos': 130,
  'token': '    ',
  'patched_gen': "I'm checking if the character is a vowel.",
  'scoped': True}]

In [ ]:
# (B) o-transfer = GoalScope replace (persistent, every step): SRC goal-head attn-out → DST   [expect T3]
# o-transfer needs no pad alignment — use the plain SRC/DST code prompts and their own pivots.
kvgen_o = o_transfer_gen(ksrc_prompt, kdst_prompt, pivot_of(ksrc_prompt), PATCH_HEADS, max_new=96)
rich_print(f"[orange3]o-transfer · DST (expect T3={T3!r}):[/orange3]\n{escape(kvgen_o)}")

print(f"\nSUMMARY (kv-section) — clean→T4({T4}) · kv-transfer→expect T2({T2}) · o-transfer→expect T3({T3})")

GoalScope[replace]:   0%|          | 0/1 [00:00<?, ?it/s]

o-transfer · DST (expect T3='count_starts_vowel'):
 b = any(w[0].lower() in 'aeiou' for w in words)
    c = sum(w[0].lower() in 'aeiou' for w in words)
    d = any(w[0].lower() in 'aeiou' for w in words)
    e = any(w[0].lower() in 'aeiou' for w in words)
    f = any(w[0].lower() in 'aeiou' for w


SUMMARY (kv-section) — clean→T4(has_palindrome) · kv-transfer→expect T2(count_long) · o-transfer→expect T3(count_starts_vowel)


: 

## Notes

- If the disentanglement holds: **q-transfer→T4, kv-transfer→T2, o-transfer→T3, clean→T2/T4**. The
  key contrast is q/kv (the *pointer*/​*menu*) vs o (the *resolved output*).
- Judge the generated first line / GoalScope readout by which predicate's computation it implements
  (oracle later if scaling up).
- Swap `T1..T5` or the datatype (`BANK = D.QTASKS[...]`) to vary; flip `PATCH_HEADS` to the top-250 set
  (cell 2) for the cleaner regime.
- o-transfer is the **exact GoalScope `strategy="replace"`** intervention, **persistent across all
  decode steps** (`n_steer_tokens="all"`): PATCH_HEADS' o_proj input is replaced with SRC's (at the
  SRC pivot) at every generated token. `O_ALPHA` = replace source-side weight (1.0 = full replace).
  `ALPHA` (OV boost) applies to the q / kv transplants (matching the eval pipeline).